In [10]:
# automatically reload each function when called for development
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from codes.PNGmodel import PNGmodel
from codes.MathModels import Y1
from codes.chain import chain
from codes.corner_plots import make_corner
from codes.helper_functions import *
from astropy.table import Table, vstack
import numpy as np
import pandas as pd

In [57]:
test = concatenate_fits(['./inputs/quadfits_LRG_FastPM_Y3_fnl_xi0.csv', 
                             './inputs/quadfits_LRG_FastPM_Y3_fnl_xi2.csv', 
                             './inputs/quadfits_LRG_FastPM_Y3_fnl_xi4.csv'])
s,c1,c2 = test

ValueError: too many values to unpack (expected 3)

In [47]:
df = pd.read_csv('./inputs/quadfits_LRG_FastPM_Y3_fnl.csv')
df = df.iloc[[0,1,2,36,37,38,72,73,74]]
df

,term,s,c1,c2
0,xi0,50.0,3.276472e-05,-2.935762e-08
1,xi0,60.0,2.606006e-05,-8.148258e-08
2,xi0,70.0,2.111597e-05,-3.499445e-08
36,xi2,50.0,-5.620066e-07,0.000000e+00
37,xi2,60.0,3.667083e-07,0.000000e+00
38,xi2,70.0,-4.588393e-07,0.000000e+00
72,xi4,50.0,0.000000e+00,0.000000e+00
73,xi4,60.0,0.000000e+00,0.000000e+00
74,xi4,70.0,0.000000e+00,0.000000e+00


In [53]:
order = ['xi2','xi0','xi4']
temp = df.sort_values(by='s')
# temp = df.sort_values(by=['term','s'], key=lambda column: column.map(lambda e: order.index(e)))
temp = df.sort_values(by='term', key=lambda column: column.map(lambda e: order.index(e)))
temp

,term,s,c1,c2
36,xi2,50.0,-5.620066e-07,0.000000e+00
37,xi2,60.0,3.667083e-07,0.000000e+00
38,xi2,70.0,-4.588393e-07,0.000000e+00
0,xi0,50.0,3.276472e-05,-2.935762e-08
2,xi0,70.0,2.111597e-05,-3.499445e-08
1,xi0,60.0,2.606006e-05,-8.148258e-08
72,xi4,50.0,0.000000e+00,0.000000e+00
73,xi4,60.0,0.000000e+00,0.000000e+00
74,xi4,70.0,0.000000e+00,0.000000e+00


In [17]:
term_masks = {}
unique_terms = list(set(df['term']))
unique_terms.sort()

for t in unique_terms:
    term_masks[t] = np.asarray(df['term']==t)

In [39]:
tab = Table.read('./inputs/abacus_averaged_fiducial_wts_fixed.fits')
tab

s,xi0,xi2,xi4
float32,float32,float32,float32
50.0,0.021023577,-0.02737209,0.0005588491
60.0,0.010780415,-0.019507943,0.00024076458
70.0,0.0055267615,-0.014327865,-5.4118827e-05
80.0,0.0030139044,-0.010381463,5.7026056e-05
90.0,0.002847832,-0.0076745367,-5.565512e-05
100.0,0.0031148214,-0.0059237923,0.000120744146
110.0,0.0019518754,-0.0047259238,0.00018823221
120.0,0.00030411332,-0.004044893,-7.8309306e-05
130.0,-0.00061912346,-0.003562457,-0.0001153505


In [43]:
test = list(tab.columns)
terms = test.copy()
terms.remove('s')
terms

['xi0', 'xi2', 'xi4']

In [20]:
with fits.open('./inputs/abacus_averaged_fiducial_wts_fixed.fits', memmap=False) as hdul:
        pkg = hdul[1].data.copy()
pkg.__dict__

{'_nfields': 4,
 '_converted': {},
 '_heapoffset': 576,
 '_heapsize': 0,
 '_tbsize': 576,
 '_gap': 0,
 '_uint': True,
 '_col_weakrefs': set(),
 '_coldefs': ColDefs(
     name = 's'; format = 'E'
     name = 'xi0'; format = 'E'
     name = 'xi2'; format = 'E'
     name = 'xi4'; format = 'E'
 ),
 '_load_variable_length_data': True,
 '_character_as_bytes': False}

In [35]:
fid = np.concatenate([pkg['xi0'], pkg['xi2'],pkg['xi4']])
mask = term_masks['xi0']
cov = np.load('./inputs/EZ_mock_covariance_matrix.npy')

cov_masked = cov[mask][:,mask]
xi_fid_masked = fid[mask]

masked = {'xi_fid':xi_fid_masked, 'cov':cov_masked}

In [31]:
masked['xi_fid']

array([ 2.10235771e-02,  1.07804146e-02,  5.52676152e-03,  3.01390444e-03,
        2.84783193e-03,  3.11482139e-03,  1.95187540e-03,  3.04113317e-04,
       -6.19123457e-04, -8.65186681e-04, -8.29161028e-04, -8.02517461e-04,
       -7.23266683e-04, -6.33441610e-04, -5.54686179e-04, -4.58485069e-04,
       -3.76285287e-04, -3.10417847e-04, -2.33922430e-04, -1.79641662e-04,
       -1.36014627e-04, -1.40383985e-04, -1.23800142e-04, -1.07598724e-04,
       -1.12706388e-04, -9.87757012e-05, -7.41978147e-05, -9.08282746e-05,
       -8.06377066e-05, -6.53163370e-05, -6.62510793e-05, -9.45791544e-05,
       -9.58757373e-05, -9.24224951e-05, -8.57750783e-05, -6.67347267e-05],
      dtype=float32)

In [32]:
# Time with masking
%timeit np.matmul(np.matmul(cov[mask][:,mask], fid[mask]), fid[mask])

11.9 μs ± 486 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [33]:
# time without masking
%timeit np.matmul(np.matmul(masked['cov'], masked['xi_fid']), masked['xi_fid'])

3.57 μs ± 57.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [36]:
# time without masking or dictionary retrieval
%timeit np.matmul(np.matmul(cov_masked, xi_fid_masked), xi_fid_masked)

3.32 μs ± 6.47 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
